In [13]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

## Tools

Tools are functions that an LLM can call to interact with the outside world (APIs, databases, search engines, etc.).

- Defined once, reused across agents and chains
- LLM decides **when** and **with what arguments** to call a tool
- Results are fed back to the LLM as context

### DuckDuckGo Search Tool

Wraps `DuckDuckGoSearchRun` with the `@tool` decorator to make it LLM-callable.

- Use for **real-time web search** (news, current events)
- No API key required
- Returns a plain-text snippet of search results

In [14]:
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def search_duckduckgo(query: str) -> str:

    """This tool searches the latest news on DuckDuckGo for the given query and returns the results."""
    duck_search = DuckDuckGoSearchRun()
    return duck_search.invoke(query)

### Arxiv Query Tool

Queries the [arXiv](https://arxiv.org) database of academic papers.

- Use for **research paper lookups** (titles, abstracts, authors)
- `ArxivAPIWrapper` handles pagination and formatting
- Returns top matching paper summaries

In [ ]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

@tool
def arxiv_tool(query: str) -> str:

    """"This tool allows you to query the arXiv database for research papers."""
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return arxiv_query.invoke(query)

### Custom Tool

Use the `@tool` decorator to turn any Python function into an LLM-callable tool.

- The **docstring** is the tool description — the LLM reads it to decide when to use the tool
- Input parameter names and types are exposed as the tool schema
- Keep docstrings clear and specific so the LLM selects the right tool

In [16]:
@tool
def personal_info(name: str) -> str:
    """Use this tool to get personal information about Alice, Bob, or Charlie."""

    info = {
        "Alice": "Alice is a software engineer with 5 years of experience in AI.",
        "Bob": "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with a background in tech startups.",
    }
    return info.get(name, f"No information available for '{name}'.")

### Tool Binding

`bind_tools` attaches a list of tools to the LLM so it knows what's available.

- The LLM returns a `tool_calls` list (not a final answer) when it wants to use a tool
- Each entry has `name`, `args`, `id`, and `type`
- You are responsible for executing the call and returning results back to the LLM

In [17]:
tools = [search_duckduckgo, arxiv_tool, personal_info]

llm_with_tools = llm.bind_tools(tools)

In [18]:
response = llm_with_tools.invoke("What are the latest news about Agentic AI?")
response.tool_calls

[{'name': 'search_duckduckgo',
  'args': {'query': 'Agentic AI'},
  'id': '0d712e3f-e49f-4edf-bde8-78520c26989c',
  'type': 'tool_call'}]